In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Day 1 (Lightweight) — Feature Engineering

Uses only existing CSVs — no re-scoring needed.

**What we add:**
- Element-wise interactions (RED-DOT-style: products and differences)
- Better XGBoost tuning (deeper trees, regularization)
- Cross-feature ratios capturing the OOC signal

**What we skip vs full version:** max/min/std aggregations (would need re-scoring)

**Total runtime:** ~1 minute

In [14]:
import os
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score

PROJECT_ROOT  = str(_cfg.ROOT)
CLIP_FEATURES = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')

# ── Load all signals ──
clip_probs = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_probs.npy'))
clip_sims  = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_sims.npy'))
id_df      = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels     = id_df['label'].values

deb_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'deberta_val_scores_v2.csv'))
deb_df['id'] = deb_df['id'].astype(str)
deb_lookup   = dict(zip(deb_df['id'], deb_df['entailment_score']))
deb_scores   = np.array([deb_lookup.get(i, 0.33) for i in id_df['id']])

ev_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'evidence_clip_scores.csv'))
ev_df['id'] = ev_df['id'].astype(str)
ev_lookup   = {row['id']: row for _, row in ev_df.iterrows()}

s2 = np.array([ev_lookup.get(i, {}).get('s2', 0.0) for i in id_df['id']])
s3 = np.array([ev_lookup.get(i, {}).get('s3', 0.0) for i in id_df['id']])
s4 = np.array([ev_lookup.get(i, {}).get('s4', 0.0) for i in id_df['id']])
s5 = np.array([ev_lookup.get(i, {}).get('s5', 0.0) for i in id_df['id']])
s6 = np.array([ev_lookup.get(i, {}).get('s6', 0.0) for i in id_df['id']])

wiki_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'wiki_nli_scores.csv'))
wiki_df['id'] = wiki_df['id'].astype(str)
wiki_lookup   = {row['id']: row for _, row in wiki_df.iterrows()}

w1 = np.array([wiki_lookup.get(i, {}).get('wiki_score',     0.33) for i in id_df['id']])
w2 = np.array([wiki_lookup.get(i, {}).get('wiki_score_min', 0.33) for i in id_df['id']])
w3 = np.array([wiki_lookup.get(i, {}).get('wiki_score_max', 0.33) for i in id_df['id']])
w4 = np.array([wiki_lookup.get(i, {}).get('entity_count',   0)    for i in id_df['id']])
w5 = np.array([wiki_lookup.get(i, {}).get('wiki_coverage',  0)    for i in id_df['id']])

print(f'All signals loaded for {len(id_df)} samples')
print(f'Real: {(labels==0).sum()} | Fake: {(labels==1).sum()}')

All signals loaded for 5000 samples
Real: 2500 | Fake: 2500


In [15]:
# ── Build engineered features ──
# RED-DOT: element-wise operations beat plain concatenation by 8.9%

eps = 1e-6

# Element-wise differences — capture image-text mismatch (the OOC signal)
diff_s2_s3   = s2 - s3        # direct: visual match minus caption match
diff_s4_s5   = s4 - s5        # inverse: same idea
diff_s2_s4   = s2 - s4        # direct vs inverse image consistency
diff_s3_s5   = s3 - s5        # direct vs inverse caption consistency

# Element-wise ratios — normalized mismatch
ratio_s3_s2  = s3 / (s2 + eps)
ratio_s5_s4  = s5 / (s4 + eps)
ratio_w_max_min = w3 / (w2 + eps)   # wiki spread

# Element-wise products — interaction terms
prod_clip_s2     = clip_probs * s2
prod_clip_s3     = clip_probs * s3
prod_clip_s6     = clip_probs * s6
prod_clipsim_s2  = clip_sims * s2
prod_clipsim_s3  = clip_sims * s3
prod_clip_wiki   = clip_probs * w1

# CLIP head disagreement
clip_diff        = clip_probs - clip_sims
clip_sum         = clip_probs + clip_sims
clip_product     = clip_probs * clip_sims

# Evidence aggregations across direct + inverse
ev_image_max     = np.maximum(s2, s4)        # best image-evidence match
ev_image_min     = np.minimum(s2, s4)        # worst image-evidence match
ev_text_max      = np.maximum(s3, s5)        # best caption-evidence match
ev_text_min      = np.minimum(s3, s5)
ev_image_text_diff = ev_image_max - ev_text_max  # the core OOC signal

# Coverage indicator — was evidence even available?
has_evidence     = ((s2 > 0) | (s4 > 0)).astype(float)

# Wiki-CLIP interactions
wiki_clip_agree  = w1 * clip_probs
wiki_evidence_agree = w1 * s3   # do wiki and direct evidence agree on caption?

# ── Final feature matrix ──
X_eng = np.column_stack([
    # Base (8)
    clip_probs, clip_sims, deb_scores,
    s2, s3, s4, s5, s6,
    # Wiki (5)
    w1, w2, w3, w4, w5,
    # Differences (4)
    diff_s2_s3, diff_s4_s5, diff_s2_s4, diff_s3_s5,
    # Ratios (3)
    ratio_s3_s2, ratio_s5_s4, ratio_w_max_min,
    # Products (6)
    prod_clip_s2, prod_clip_s3, prod_clip_s6,
    prod_clipsim_s2, prod_clipsim_s3, prod_clip_wiki,
    # CLIP head ops (3)
    clip_diff, clip_sum, clip_product,
    # Evidence aggregations (5)
    ev_image_max, ev_image_min, ev_text_max, ev_text_min, ev_image_text_diff,
    # Coverage (1)
    has_evidence,
    # Wiki interactions (2)
    wiki_clip_agree, wiki_evidence_agree,
])

feature_names = [
    'clip_prob', 'clip_sim', 'deberta',
    's2', 's3', 's4', 's5', 's6',
    'wiki_mean', 'wiki_min', 'wiki_max', 'entity_cnt', 'wiki_cov',
    'diff_s2_s3', 'diff_s4_s5', 'diff_s2_s4', 'diff_s3_s5',
    'ratio_s3_s2', 'ratio_s5_s4', 'ratio_wiki_maxmin',
    'clip_x_s2', 'clip_x_s3', 'clip_x_s6',
    'clipsim_x_s2', 'clipsim_x_s3', 'clip_x_wiki',
    'clip_diff', 'clip_sum', 'clip_product',
    'ev_img_max', 'ev_img_min', 'ev_txt_max', 'ev_txt_min', 'ev_img_txt_diff',
    'has_evidence',
    'wiki_x_clip', 'wiki_x_s3',
]

print(f'Feature matrix: {X_eng.shape}')
print(f'Features: {len(feature_names)}')
assert X_eng.shape[1] == len(feature_names), 'Mismatch in feature names!'

Feature matrix: (5000, 37)
Features: 37


In [16]:
# ── Train tuned XGBoost ──
X_train, X_val, y_train, y_val = train_test_split(
    X_eng, labels, test_size=0.2, random_state=42, stratify=labels)

xgb = XGBClassifier(
    n_estimators=1500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='logloss',
    early_stopping_rounds=50,
    random_state=42,
)

xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

preds = xgb.predict(X_val)
probs = xgb.predict_proba(X_val)[:, 1]

acc = accuracy_score(y_val, preds)
f1  = f1_score(y_val, preds)
auc = roc_auc_score(y_val, probs)

print()
print('=' * 65)
print('XGBoost — Engineered Features')
print('=' * 65)
print(f'Accuracy : {acc*100:.2f}%')
print(f'F1       : {f1:.4f}')
print(f'AUC      : {auc:.4f}')
print()
print(classification_report(y_val, preds, target_names=['REAL', 'FAKE']))

[0]	validation_0-logloss:0.67487
[100]	validation_0-logloss:0.29380
[200]	validation_0-logloss:0.29215
[210]	validation_0-logloss:0.29315

XGBoost — Engineered Features
Accuracy : 86.20%
F1       : 0.8628
AUC      : 0.9482

              precision    recall  f1-score   support

        REAL       0.87      0.86      0.86       500
        FAKE       0.86      0.87      0.86       500

    accuracy                           0.86      1000
   macro avg       0.86      0.86      0.86      1000
weighted avg       0.86      0.86      0.86      1000



In [17]:
# ── Feature importance breakdown ──
imp_df = pd.DataFrame({
    'feature'   : feature_names,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 15 features:')
print(imp_df.head(15).to_string(index=False))

# Group by category
categories = {
    'CLIP base'      : ['clip_prob', 'clip_sim'],
    'Text signals'   : ['deberta', 'wiki_mean', 'wiki_min', 'wiki_max', 'entity_cnt', 'wiki_cov'],
    'Evidence raw'   : ['s2', 's3', 's4', 's5', 's6'],
    'Differences'    : ['diff_s2_s3', 'diff_s4_s5', 'diff_s2_s4', 'diff_s3_s5'],
    'Ratios'         : ['ratio_s3_s2', 'ratio_s5_s4', 'ratio_wiki_maxmin'],
    'Products'       : ['clip_x_s2', 'clip_x_s3', 'clip_x_s6',
                         'clipsim_x_s2', 'clipsim_x_s3', 'clip_x_wiki',
                         'clip_diff', 'clip_sum', 'clip_product',
                         'wiki_x_clip', 'wiki_x_s3'],
    'Evidence agg'   : ['ev_img_max', 'ev_img_min', 'ev_txt_max', 'ev_txt_min', 'ev_img_txt_diff'],
    'Meta'           : ['has_evidence'],
}

print()
print('Importance by category:')
for cat, feats in categories.items():
    total = imp_df[imp_df['feature'].isin(feats)]['importance'].sum()
    n = len(feats)
    print(f'  {cat:<15} ({n:2d} feats): {total:.4f}')

Top 15 features:
     feature  importance
   clip_diff    0.308281
   clip_prob    0.183697
    clip_sum    0.050674
          s5    0.034464
    clip_sim    0.025047
   clip_x_s2    0.021638
   clip_x_s3    0.021011
  diff_s4_s5    0.020616
clip_product    0.019915
  ev_txt_min    0.018825
  ev_img_min    0.015972
          s4    0.015446
          s6    0.014442
          s3    0.012469
  ev_txt_max    0.012340

Importance by category:
  CLIP base       ( 2 feats): 0.2087
  Text signals    ( 6 feats): 0.0611
  Evidence raw    ( 5 feats): 0.0879
  Differences     ( 4 feats): 0.0544
  Ratios          ( 3 feats): 0.0322
  Products        (11 feats): 0.4879
  Evidence agg    ( 5 feats): 0.0677
  Meta            ( 1 feats): 0.0000


In [18]:
# ── Progression vs literature ──
print('=' * 65)
print('PROGRESSION')
print('=' * 65)
results = [
    ('Your CLIP alone (v2)',                  85.60),
    ('+ DeBERTa + Evidence (mean only, MLP)', 87.40),
    ('+ XGBoost on 8 features',                87.30),
    ('+ Wiki NLI (no engineering)',            87.20),
    ('+ Engineered features (Day 1)',          acc * 100),
    ('--- Targets ---',                         0),
    ('SNIFFER',                                88.40),
    ('MUSE-MLP',                               90.00),
    ('RED-DOT',                                90.30),
    ('MUSE-AITR',                              93.30),
]
for name, val in results:
    if val == 0:
        print(f'  {name}')
    else:
        marker = ' <-- YOU' if 'Day 1' in name else ''
        print(f'  {name:<42} {val:.2f}%{marker}')

PROGRESSION
  Your CLIP alone (v2)                       85.60%
  + DeBERTa + Evidence (mean only, MLP)      87.40%
  + XGBoost on 8 features                    87.30%
  + Wiki NLI (no engineering)                87.20%
  + Engineered features (Day 1)              86.20% <-- YOU
  --- Targets ---
  SNIFFER                                    88.40%
  MUSE-MLP                                   90.00%
  RED-DOT                                    90.30%
  MUSE-AITR                                  93.30%


In [19]:
# ── Optional: try LightGBM and CatBoost for an ensemble ──
# Different gradient boosters often disagree on different samples — averaging helps

from sklearn.ensemble import GradientBoostingClassifier

models = {
    'XGBoost'         : xgb,
}

# Try LightGBM if available
try:
    from lightgbm import LGBMClassifier
    lgbm = LGBMClassifier(n_estimators=1500, max_depth=5, learning_rate=0.03,
                          subsample=0.85, colsample_bytree=0.85,
                          reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)
    lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[])
    models['LightGBM'] = lgbm
except ImportError:
    print('LightGBM not installed — skipping (pip install lightgbm)')

# Try CatBoost if available
try:
    from catboost import CatBoostClassifier
    cb = CatBoostClassifier(iterations=1500, depth=5, learning_rate=0.03,
                             l2_leaf_reg=3, random_seed=42, verbose=False,
                             early_stopping_rounds=50)
    cb.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)
    models['CatBoost'] = cb
except ImportError:
    print('CatBoost not installed — skipping (pip install catboost)')

# Individual + ensemble
all_probs = []
print()
for name, model in models.items():
    p = model.predict_proba(X_val)[:, 1]
    all_probs.append(p)
    pred = (p > 0.5).astype(int)
    print(f'{name:<10} : Acc={accuracy_score(y_val, pred)*100:.2f}%  F1={f1_score(y_val, pred):.4f}')

if len(all_probs) > 1:
    ens_probs = np.mean(all_probs, axis=0)
    ens_preds = (ens_probs > 0.5).astype(int)
    print(f'{"Ensemble":<10} : Acc={accuracy_score(y_val, ens_preds)*100:.2f}%  F1={f1_score(y_val, ens_preds):.4f}  AUC={roc_auc_score(y_val, ens_probs):.4f}')

LightGBM not installed — skipping (pip install lightgbm)
CatBoost not installed — skipping (pip install catboost)

XGBoost    : Acc=86.20%  F1=0.8628
